# LeetCode 3336 — Find the Number of Subsequences With Equal GCD bằng Python

- Link: https://leetcode.com/problems/find-the-number-of-subsequences-with-equal-gcd/
- Độ khó: **Hard**
- Chủ đề: **DP trạng thái (gcd1, gcd2)** — mỗi phần tử vào dãy 1, dãy 2 hoặc bỏ; **nghịch đảo Möbius 2 chiều** để đếm "gcd đúng bằng `g`" từ "gcd chia hết cho `a`, `b`"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def subsequencePairCount(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Mô hình hóa cặp dãy con **rời nhau** = gán mỗi chỉ số một trong 3 nhãn: dãy 1 / dãy 2 / không chọn.
2. DP với trạng thái `(g1, g2)` = gcd hiện tại của hai dãy (quy ước `gcd` của dãy rỗng là `0`, vì `gcd(0, x) = x`).
3. Tối ưu DP bằng **từ điển chỉ chứa trạng thái đạt được** (gcd giảm rất nhanh).
4. Cách nhìn số học: đếm cặp mà mọi phần tử dãy 1 chia hết cho `a`, dãy 2 chia hết cho `b` bằng công thức `3^{both} · 2^{onlyA} · 2^{onlyB}`, rồi **Möbius 2 chiều** để ra "gcd đúng bằng `g`".
5. Kiểm chứng bằng vét cạn `3^n` cách gán.

## 2. Đề bài & ví dụ

Cho mảng `nums`. Đếm số **cặp có thứ tự** `(seq1, seq2)` các dãy con **khác rỗng**, **rời nhau** (không chung chỉ số), có `gcd(seq1) = gcd(seq2)`. Modulo `10^9 + 7`.

**Ví dụ 1:** `[1,2,3,4]` → `10`. **Ví dụ 2:** `[10,20,30]` → `2`. **Ví dụ 3:** `[1,1,1,1]` → `50`.

**Ràng buộc:** `1 ≤ n ≤ 200`, `1 ≤ nums[i] ≤ 200`.

In [1]:
from typing import List
from math import gcd
from itertools import product
MOD = 10 ** 9 + 7

examples = [([1, 2, 3, 4], 10), ([10, 20, 30], 2), ([1, 1, 1, 1], 50)]
for nums, exp in examples:
    print(nums, '-> kỳ vọng', exp)

[1, 2, 3, 4] -> kỳ vọng 10
[10, 20, 30] -> kỳ vọng 2
[1, 1, 1, 1] -> kỳ vọng 50


## 3. Kiến thức nền

### 3.1. Gán 3 nhãn

Một cặp `(seq1, seq2)` rời nhau tương ứng **một-một** với một cách gán mỗi chỉ số nhãn `1`, `2` hoặc `0` (không chọn). Có `3^n` cách gán; yêu cầu thêm: cả hai nhãn đều được dùng và hai gcd bằng nhau.

### 3.2. gcd của dãy rỗng

`gcd(0, x) = x` nên dùng `0` làm gcd của dãy rỗng — cập nhật `g ← gcd(g, x)` không cần xử lý riêng; cuối cùng loại các trạng thái có `g1 = 0` hoặc `g2 = 0`.

In [2]:
print(gcd(0, 12), gcd(gcd(0, 12), 18), gcd(0, 0))

12 6 0


### 3.3. Hàm Möbius và nghịch đảo

`μ(1) = 1`, `μ(n) = 0` nếu `n` có ước chính phương, `μ(n) = (−1)^k` nếu `n` là tích `k` số nguyên tố phân biệt. Nếu `F(a) = Σ_{a | d} f(d)` thì `f(g) = Σ_{i ≥ 1} μ(i) F(g·i)`. Áp dụng **hai lần** (cho hai chiều `a`, `b`).

In [3]:
def mobius_table(M):
    mu = [1] * (M + 1)
    is_prime = [True] * (M + 1)
    for p in range(2, M + 1):
        if is_prime[p]:
            for q in range(2 * p, M + 1, p):
                is_prime[q] = False
            for q in range(p, M + 1, p):
                mu[q] = -mu[q]
            for q in range(p * p, M + 1, p * p):
                mu[q] = 0
    return mu

print('mu[1..12] =', mobius_table(12)[1:])

mu[1..12] = [1, -1, -1, 0, -1, 1, -1, 0, 0, 1, -1, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

`[10, 20, 30]`: muốn `gcd(seq1) = gcd(seq2)`. Cặp `({10}, {20})`? gcd `10 ≠ 20`. `({10}, {20, 30})`: `10 = gcd(20, 30) = 10` ✓; đảo lại `({20, 30}, {10})` ✓. Ngoài ra không còn cặp nào ⇒ `2` ✓.

`[1, 1, 1, 1]`: mọi cặp khác rỗng rời nhau đều có gcd `1 = 1` ⇒ số cách gán 3 nhãn trừ các cách thiếu nhãn: `3^4 − 2·2^4 + 1 = 81 − 32 + 1 = 50` ✓.

## 5. Cách 1 — Brute Force: mọi cách gán 3 nhãn

In [4]:
def brute(nums):
    total = 0
    for labels in product(range(3), repeat=len(nums)):
        g = [0, 0, 0]
        for x, lab in zip(nums, labels):
            g[lab] = gcd(g[lab], x)
        if g[1] and g[1] == g[2]:
            total += 1
    return total % MOD

for nums, exp in examples:
    print(brute(nums), '| kỳ vọng', exp)

10 | kỳ vọng 10
2 | kỳ vọng 2
50 | kỳ vọng 50


**Độ phức tạp:** `O(3^n · n)` — chỉ dùng cho `n ≤ 10`.

## 6. Ý tưởng thuật toán / bước đột phá

**DP:** `dp[(g1, g2)]` = số cách gán các phần tử đã xét dẫn tới gcd hai dãy là `g1`, `g2`. Với mỗi `x`:

- không chọn: giữ nguyên `(g1, g2)`;
- vào dãy 1: `(gcd(g1, x), g2)`;
- vào dãy 2: `(g1, gcd(g2, x))`.

Đáp án `= Σ_{g ≥ 1} dp[(g, g)]`. Trạng thái tối đa `201²`, nhưng thực tế số trạng thái **đạt được** nhỏ hơn nhiều.

**Möbius (nhanh hơn hẳn):** gọi `C(a, b)` = số cặp (khác rỗng, rời nhau) mà mọi phần tử dãy 1 chia hết cho `a`, mọi phần tử dãy 2 chia hết cho `b`. Đếm theo từng phần tử: chia hết cho cả `a` và `b` (tức cho `lcm(a, b)`) ⇒ 3 lựa chọn; chỉ chia hết cho `a` ⇒ 2 lựa chọn; chỉ cho `b` ⇒ 2; còn lại ⇒ 1. Trừ các trường hợp một dãy rỗng (bao hàm–loại trừ). Sau đó `E(g) = Σ_{i, j} μ(i) μ(j) C(g·i, g·j)` là số cặp có cả hai gcd **đúng bằng** `g`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP mảng `201 × 201` (bỏ qua ô bằng 0)

In [5]:
def dp_array(nums):
    M = max(nums)
    G = [[gcd(a, b) for b in range(M + 1)] for a in range(M + 1)]
    dp = [[0] * (M + 1) for _ in range(M + 1)]
    dp[0][0] = 1
    for x in nums:
        nd = [row[:] for row in dp]                  # nhãn 0: không chọn
        Gx = [G[a][x] for a in range(M + 1)]
        for a in range(M + 1):
            row = dp[a]
            ga = Gx[a]
            for b in range(M + 1):
                v = row[b]
                if v:
                    nd[ga][b] = (nd[ga][b] + v) % MOD        # vào dãy 1
                    gb = Gx[b]
                    nd[a][gb] = (nd[a][gb] + v) % MOD        # vào dãy 2
        dp = nd
    return sum(dp[g][g] for g in range(1, M + 1)) % MOD

for nums, exp in examples:
    print(dp_array(nums), '| kỳ vọng', exp)

10 | kỳ vọng 10
2 | kỳ vọng 2
50 | kỳ vọng 50


### 7.2. Cách 3 — DP từ điển (chỉ trạng thái đạt được)

In [6]:
def dp_dict(nums):
    dp = {(0, 0): 1}
    for x in nums:
        nd = dict(dp)
        for (a, b), v in dp.items():
            k = (gcd(a, x), b)
            nd[k] = (nd.get(k, 0) + v) % MOD
            k = (a, gcd(b, x))
            nd[k] = (nd.get(k, 0) + v) % MOD
        dp = nd
    return sum(v for (a, b), v in dp.items() if a and a == b) % MOD

for nums, exp in examples:
    print(dp_dict(nums), '| kỳ vọng', exp)

10 | kỳ vọng 10
2 | kỳ vọng 2
50 | kỳ vọng 50


### 7.3. Cách 4 — Đếm "chia hết" + Möbius 2 chiều (lời giải chính)

In [7]:
def mobius_solution(nums):
    n, M = len(nums), max(nums)
    mu = mobius_table(M)
    freq = [0] * (M + 1)
    for x in nums:
        freq[x] += 1
    cd = [0] * (M + 1)                                 # cd[v] = số phần tử chia hết cho v
    for v in range(1, M + 1):
        cd[v] = sum(freq[v::v])
    p2 = [1] * (n + 1); p3 = [1] * (n + 1)
    for i in range(1, n + 1):
        p2[i] = p2[i - 1] * 2 % MOD
        p3[i] = p3[i - 1] * 3 % MOD

    def C(a, b):
        l = a * b // gcd(a, b)
        both = cd[l] if l <= M else 0
        only_a, only_b = cd[a] - both, cd[b] - both
        total = p3[both] * p2[only_a] % MOD * p2[only_b] % MOD
        return (total - p2[cd[b]] - p2[cd[a]] + 1) % MOD   # trừ dãy 1 rỗng, dãy 2 rỗng, cộng lại cả hai rỗng

    ans = 0
    for g in range(1, M + 1):
        K = M // g
        for i in range(1, K + 1):
            if mu[i]:
                for j in range(1, K + 1):
                    if mu[j]:
                        ans += mu[i] * mu[j] * C(g * i, g * j)
    return ans % MOD

for nums, exp in examples:
    print(mobius_solution(nums), '| kỳ vọng', exp)

10 | kỳ vọng 10
2 | kỳ vọng 2
50 | kỳ vọng 50


Số cặp `(g, i, j)`: `Σ_g (M/g)² ≈ M² · π²/6 ≈ 6.6·10^4` với `M = 200` — rất nhanh.

## 8. TRACE từng bước

Trace cách 3 trên `[10, 20, 30]` (in các trạng thái `(g1, g2): số cách`).

In [8]:
def trace(nums):
    dp = {(0, 0): 1}
    for x in nums:
        nd = dict(dp)
        for (a, b), v in dp.items():
            for k in ((gcd(a, x), b), (a, gcd(b, x))):
                nd[k] = nd.get(k, 0) + v
        dp = nd
        print(f'sau x = {x}:', dict(sorted(dp.items())))
    print('trạng thái (g, g) với g >= 1:', {k: v for k, v in dp.items() if k[0] and k[0] == k[1]})
    print('đáp án:', dp_dict(nums), '| Möbius:', mobius_solution(nums), '| vét cạn:', brute(nums))

trace([10, 20, 30])

sau x = 10: {(0, 0): 1, (0, 10): 1, (10, 0): 1}
sau x = 20: {(0, 0): 1, (0, 10): 2, (0, 20): 1, (10, 0): 2, (10, 20): 1, (20, 0): 1, (20, 10): 1}
sau x = 30: {(0, 0): 1, (0, 10): 5, (0, 20): 1, (0, 30): 1, (10, 0): 5, (10, 10): 2, (10, 20): 2, (10, 30): 2, (20, 0): 1, (20, 10): 2, (20, 30): 1, (30, 0): 1, (30, 10): 2, (30, 20): 1}
trạng thái (g, g) với g >= 1: {(10, 10): 2}
đáp án: 2 | Möbius: 2 | vét cạn: 2


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def subsequencePairCount(self, nums: List[int]) -> int:
        MOD = 10 ** 9 + 7
        n, M = len(nums), max(nums)
        mu = [1] * (M + 1)
        sieve = [True] * (M + 1)
        for p in range(2, M + 1):
            if sieve[p]:
                for q in range(2 * p, M + 1, p):
                    sieve[q] = False
                for q in range(p, M + 1, p):
                    mu[q] = -mu[q]
                for q in range(p * p, M + 1, p * p):
                    mu[q] = 0
        freq = [0] * (M + 1)
        for x in nums:
            freq[x] += 1
        cd = [0] * (M + 1)
        for v in range(1, M + 1):
            cd[v] = sum(freq[v::v])
        p2 = [1] * (n + 1)
        p3 = [1] * (n + 1)
        for i in range(1, n + 1):
            p2[i] = p2[i - 1] * 2 % MOD
            p3[i] = p3[i - 1] * 3 % MOD
        ans = 0
        for g in range(1, M + 1):
            if cd[g] < 2:                      # cần ít nhất 2 phần tử chia hết cho g
                continue
            K = M // g
            for i in range(1, K + 1):
                if not mu[i]:
                    continue
                a = g * i
                for j in range(1, K + 1):
                    if not mu[j]:
                        continue
                    b = g * j
                    l = a // gcd(a, b) * b
                    both = cd[l] if l <= M else 0
                    total = p3[both] * p2[cd[a] - both] % MOD * p2[cd[b] - both] % MOD
                    c = total - p2[cd[b]] - p2[cd[a]] + 1
                    ans += mu[i] * mu[j] * c
        return ans % MOD

for nums, exp in examples:
    got = Solution().subsequencePairCount(nums)
    print(got, 'OK' if got == exp else 'SAI')

10 OK
2 OK
50 OK


Cắt tỉa `cd[g] < 2`: cả hai dãy khác rỗng, rời nhau, mọi phần tử chia hết cho `g` ⇒ cần ít nhất 2 phần tử như vậy; khi đó mọi `C(g·i, g·j)` đều bằng 0.

## 10. Vì sao thuật toán đúng

**DP:** song ánh giữa cặp rời nhau và cách gán 3 nhãn (mục 3.1); trạng thái `(g1, g2)` đủ thông tin vì gcd của dãy sau khi thêm `x` chỉ phụ thuộc gcd cũ và `x`.

**Möbius:** đặt `f(g1, g2)` = số cặp có gcd đúng bằng `(g1, g2)`. Một cặp có mọi phần tử dãy 1 chia hết cho `a` ⇔ `a | gcd(seq1)`. Vậy `C(a, b) = Σ_{a | g1, b | g2} f(g1, g2)`. Nghịch đảo Möbius trên từng chiều cho `f(g, g) = Σ_{i, j} μ(i) μ(j) C(g i, g j)` (chỉ cần các bội `≤ M`, vì `C = 0` khi `a > M` hay `b > M` — không phần tử nào chia hết).

**Công thức `C(a, b)`:** phần tử chia hết cho cả `a` và `b` (⇔ chia hết cho `lcm`) được tự do 3 nhãn; chỉ chia hết cho `a` thì nhãn `1` hoặc `0`; chỉ chia hết cho `b` thì `2` hoặc `0`; còn lại bắt buộc `0`. Trừ các gán có dãy 1 rỗng (`2^{cd[b]}`), dãy 2 rỗng (`2^{cd[a]}`), cộng lại gán cả hai rỗng (1).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(3^n · n)` | `O(n)` |
| 2. DP mảng | `O(n · M²)` ≈ `8·10^6` | `O(M²)` |
| 3. DP từ điển | `O(n · S)`, `S` = số trạng thái đạt được | `O(S)` |
| 4. Möbius | `O(M log M + M² · π²/6)` | `O(M)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'dp_array': dp_array, 'dp_dict': dp_dict, 'mobius': mobius_solution,
         'submit': lambda a: Solution().subsequencePairCount(a)}
edge = [([5], 0), ([7, 7], 2), ([2, 3], 0), ([2, 4], 0), ([6, 10, 15], 0), ([4, 6, 2], 2)]
for nums, exp in examples + edge:
    assert brute(nums) == exp, (nums, brute(nums))
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums, f(nums))
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3336)
cnt = 0
for _ in range(400):
    nums = [rng.randint(1, rng.choice([6, 30, 200])) for _ in range(rng.randint(1, 8))]
    exp = brute(nums)
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'mảng ngẫu nhiên.')

Khớp vét cạn trên 400 mảng ngẫu nhiên.


### 12.3. Ca lớn `n = 200`, giá trị tới 200

In [12]:
import time
cases = {
    'ngẫu nhiên 1..200': [rng.randint(1, 200) for _ in range(200)],
    'nhiều ước chung': [rng.choice([60, 72, 84, 96, 120, 144, 150, 160, 168, 180, 192, 200]) for _ in range(200)],
    '1..200': list(range(1, 201)),
    'toàn 200': [200] * 200,
}
for name, nums in cases.items():
    t = time.time(); a = Solution().subsequencePairCount(nums); t1 = time.time() - t
    t = time.time(); b = dp_dict(nums); t2 = time.time() - t
    assert a == b
    print(f"{name:18s}: {a:10d} | Möbius {t1:.3f}s | DP từ điển {t2:.2f}s")
t = time.time(); assert dp_array(cases['nhiều ước chung'][:60]) == mobius_solution(cases['nhiều ước chung'][:60]); print(f'DP mảng (n = 60): {time.time() - t:.2f}s')

ngẫu nhiên 1..200 :  246860549 | Möbius 0.008s | DP từ điển 0.79s
nhiều ước chung   :  319413213 | Möbius 0.007s | DP từ điển 0.06s


1..200            :  246467506 | Möbius 0.008s | DP từ điển 1.20s
toàn 200          :  137428029 | Möbius 0.009s | DP từ điển 0.00s
DP mảng (n = 60): 0.07s


Kiểm tra độc lập: mảng toàn cùng một giá trị ⇒ mọi cặp hợp lệ ⇒ `3^n − 2·2^n + 1`.

In [13]:
n = 200
assert Solution().subsequencePairCount([200] * n) == (pow(3, n, MOD) - 2 * pow(2, n, MOD) + 1) % MOD
print('toàn 200: khớp 3^n - 2·2^n + 1')

toàn 200: khớp 3^n - 2·2^n + 1


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm cặp **không thứ tự** (chia đôi)

In [14]:
def wrong_unordered(nums):
    return dp_dict(nums) * pow(2, MOD - 2, MOD) % MOD        # SAI: đề đếm cặp có thứ tự

print('sai :', wrong_unordered([10, 20, 30]), '| đúng:', Solution().subsequencePairCount([10, 20, 30]))

sai : 1 | đúng: 2


**Vì sao sai:** ví dụ 2 liệt kê cả `({10}, {20, 30})` và `({20, 30}, {10})`.

### Lỗi 2 — Cho phép dãy rỗng (gcd `0 = 0`)

In [15]:
def wrong_allow_empty(nums):
    dp = {(0, 0): 1}
    for x in nums:
        nd = dict(dp)
        for (a, b), v in dp.items():
            for k in ((gcd(a, x), b), (a, gcd(b, x))):
                nd[k] = (nd.get(k, 0) + v) % MOD
        dp = nd
    return sum(v for (a, b), v in dp.items() if a == b) % MOD   # SAI: tính cả (0, 0)

print('sai :', wrong_allow_empty([2, 3]), '| đúng:', Solution().subsequencePairCount([2, 3]))

sai : 1 | đúng: 0


**Vì sao sai:** trạng thái `(0, 0)` là "cả hai dãy rỗng" — luôn có đúng 1 cách (không chọn gì), đề yêu cầu khác rỗng.

### Lỗi 3 — Quên điều kiện rời nhau (đếm độc lập rồi bình phương)

In [16]:
def wrong_not_disjoint(nums):
    by_gcd = {}
    for mask in range(1, 1 << len(nums)):
        g = 0
        for i, x in enumerate(nums):
            if mask >> i & 1:
                g = gcd(g, x)
        by_gcd[g] = by_gcd.get(g, 0) + 1
    return sum(c * c for c in by_gcd.values()) % MOD          # SAI: hai dãy có thể dùng chung chỉ số

print('sai :', wrong_not_disjoint([10, 20, 30]), '| đúng:', Solution().subsequencePairCount([10, 20, 30]))

sai : 27 | đúng: 2


**Vì sao sai:** cặp `({10}, {10})` (cùng chỉ số) bị tính, trong khi đề yêu cầu không chung chỉ số.

### Lỗi 4 — Möbius nhưng quên trừ các trường hợp một dãy rỗng trong `C(a, b)`

In [17]:
def wrong_C_no_empty_fix(nums):
    n, M = len(nums), max(nums)
    mu = mobius_table(M)
    cd = [0] * (M + 1)
    for v in range(1, M + 1):
        cd[v] = sum(1 for x in nums if x % v == 0)
    ans = 0
    for g in range(1, M + 1):
        for i in range(1, M // g + 1):
            for j in range(1, M // g + 1):
                a, b = g * i, g * j
                l = a * b // gcd(a, b)
                both = cd[l] if l <= M else 0
                ans += mu[i] * mu[j] * 3 ** both * 2 ** (cd[a] - both) * 2 ** (cd[b] - both)   # SAI
    return ans % MOD

print('sai :', wrong_C_no_empty_fix([1, 1, 1, 1]), '| đúng:', Solution().subsequencePairCount([1, 1, 1, 1]))

sai : 81 | đúng: 50


**Vì sao sai:** `3^{both}…` đếm cả gán mà dãy 1 hoặc dãy 2 rỗng; gcd dãy rỗng (= 0) "chia hết" cho mọi `a` nên chúng lọt qua cả nghịch đảo Möbius.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cặp dãy con rời nhau" ⇒ gán 3 nhãn cho mỗi phần tử.
- "gcd đúng bằng `g`" ⇒ đếm "gcd là bội của `g`" (dễ: chỉ cần mọi phần tử chia hết) rồi Möbius / bao hàm–loại trừ theo bội.
- DP trên gcd: số giá trị gcd đạt được nhỏ ⇒ dùng từ điển.

## 15. Biến thể của bài

1. Đếm cặp có `gcd(seq1) = 2 · gcd(seq2)` ⇒ cộng `f(2g, g)`.
2. Đếm **bộ ba** dãy con rời nhau cùng gcd ⇒ 4 nhãn, Möbius 3 chiều.
3. Không yêu cầu rời nhau ⇒ `Σ_g h(g)²` với `h(g)` = số dãy con có gcd `g`.
4. Chỉ đếm số dãy con có gcd `g` (một chiều) — LC "Number of Different Subsequences GCDs" (1819) cùng họ.

In [18]:
# Biến thể 1 và 3
def gcd_ratio_two(nums):
    dp = {(0, 0): 1}
    for x in nums:
        nd = dict(dp)
        for (a, b), v in dp.items():
            for k in ((gcd(a, x), b), (a, gcd(b, x))):
                nd[k] = (nd.get(k, 0) + v) % MOD
        dp = nd
    return sum(v for (a, b), v in dp.items() if b and a == 2 * b) % MOD

def not_disjoint(nums):
    M = max(nums)
    mu = mobius_table(M)
    cd = [0] + [sum(1 for x in nums if x % v == 0) for v in range(1, M + 1)]
    h = [0] * (M + 1)
    for g in range(1, M + 1):
        h[g] = sum(mu[i] * (pow(2, cd[g * i], MOD) - 1) for i in range(1, M // g + 1)) % MOD
    return sum(v * v for v in h) % MOD

rng = random.Random(15)
for _ in range(200):
    nums = [rng.randint(1, 24) for _ in range(rng.randint(1, 7))]
    exp1 = 0
    for labels in product(range(3), repeat=len(nums)):
        g = [0, 0, 0]
        for x, lab in zip(nums, labels):
            g[lab] = gcd(g[lab], x)
        exp1 += bool(g[2]) and g[1] == 2 * g[2]
    assert gcd_ratio_two(nums) == exp1
    assert not_disjoint(nums) == wrong_not_disjoint(nums)
print('Biến thể 1 khớp vét cạn; biến thể 3 (không rời nhau) bằng Möbius khớp liệt kê.')

Biến thể 1 khớp vét cạn; biến thể 3 (không rời nhau) bằng Möbius khớp liệt kê.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Gán 3 nhãn | `O(3^n n)` | Oracle |
| 2 | DP mảng `(g1, g2)` | `O(n M²)` | Lời giải "chuẩn" (chậm trong Python) |
| 3 | DP từ điển | `O(n S)` | Nhanh trong thực tế |
| 4 | Möbius 2 chiều | `O(M²)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[3, 3, 3]`.

**Bài 2 (tính tay):** `[2, 4, 6]`.

**Bài 3 (điền TODO):** trả về số cặp theo **từng** giá trị `g` (từ điển `g → số cặp`).

**Bài 4 (tự viết lại):** biến thể 2 — bộ ba dãy con rời nhau cùng gcd (vét cạn + DP từ điển 3 chiều).

**Bài 5 (mini challenge):** cặp rời nhau với `gcd(seq1) + gcd(seq2) = K`.

In [19]:
def per_g_todo(nums):
    # TODO
    raise NotImplementedError

def triples_todo(nums):
    # TODO
    raise NotImplementedError

def gcd_sum_k_todo(nums, K):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mọi cặp hợp lệ. (2) `3^3 − 2·2^3 + 1`. (3) `12`.

**Bài 2.** (1) gcd chung chỉ có thể là `2` (gcd 4, 6 không đạt được ở cả hai phía cùng lúc). (2) Dãy có gcd 2: `{2}`, `{2,4}`, `{2,6}`, `{4,6}`, `{2,4,6}` — cặp rời nhau chỉ có `{2}` với `{4,6}`. (3) Tính cả thứ tự: `2`.

**Bài 3.** (1) DP từ điển, gom `dp[(g, g)]`. (2) Hoặc Möbius theo từng `g`. (3) Tổng phải bằng đáp án.

**Bài 4.** (1) Trạng thái `(g1, g2, g3)`. (2) 4 lựa chọn mỗi phần tử. (3) Vét cạn `4^n`.

**Bài 5.** (1) Lấy toàn bộ bảng `dp`. (2) Cộng `dp[(a, b)]` với `a, b ≥ 1`, `a + b = K`. (3) Vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().subsequencePairCount([3, 3, 3]) == 12
assert Solution().subsequencePairCount([2, 4, 6]) == 2 == brute([2, 4, 6])

def per_g(nums):
    dp = {(0, 0): 1}
    for x in nums:
        nd = dict(dp)
        for (a, b), v in dp.items():
            for k in ((gcd(a, x), b), (a, gcd(b, x))):
                nd[k] = nd.get(k, 0) + v
        dp = nd
    return {a: v for (a, b), v in dp.items() if a and a == b}

rng = random.Random(3)
for _ in range(200):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(1, 8))]
    assert sum(per_g(nums).values()) % MOD == Solution().subsequencePairCount(nums)
print('Bài 1: 12 | Bài 2: 2 | Bài 3: tổng theo g khớp đáp án. Ví dụ per_g([2, 4, 6]) =', per_g([2, 4, 6]))

Bài 1: 12 | Bài 2: 2 | Bài 3: tổng theo g khớp đáp án. Ví dụ per_g([2, 4, 6]) = {2: 2}


In [21]:
# Bài 4, 5
def triples(nums):
    dp = {(0, 0, 0): 1}
    for x in nums:
        nd = dict(dp)
        for st, v in dp.items():
            for t in range(3):
                k = list(st); k[t] = gcd(k[t], x); k = tuple(k)
                nd[k] = (nd.get(k, 0) + v) % MOD
        dp = nd
    return sum(v for (a, b, c), v in dp.items() if a and a == b == c) % MOD

def gcd_sum_k(nums, K):
    dp = {(0, 0): 1}
    for x in nums:
        nd = dict(dp)
        for (a, b), v in dp.items():
            for k in ((gcd(a, x), b), (a, gcd(b, x))):
                nd[k] = (nd.get(k, 0) + v) % MOD
        dp = nd
    return sum(v for (a, b), v in dp.items() if a and b and a + b == K) % MOD

rng = random.Random(4)
for _ in range(150):
    nums = [rng.randint(1, 12) for _ in range(rng.randint(1, 6))]
    K = rng.randint(2, 20)
    e3 = e5 = 0
    for labels in product(range(4), repeat=len(nums)):
        g = [0, 0, 0, 0]
        for x, lab in zip(nums, labels):
            g[lab] = gcd(g[lab], x)
        e3 += bool(g[1]) and g[1] == g[2] == g[3]
    for labels in product(range(3), repeat=len(nums)):
        g = [0, 0, 0]
        for x, lab in zip(nums, labels):
            g[lab] = gcd(g[lab], x)
        e5 += bool(g[1]) and bool(g[2]) and g[1] + g[2] == K
    assert triples(nums) == e3 and gcd_sum_k(nums, K) == e5
print('Bài 4, 5 khớp vét cạn.')

Bài 4, 5 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cd[v] = #phần tử chia hết cho v
C(a,b) = 3^both · 2^(cd[a]-both) · 2^(cd[b]-both) − 2^cd[b] − 2^cd[a] + 1   (both = cd[lcm])
ans = Σ_g Σ_{i,j ≤ M/g} μ(i) μ(j) C(g·i, g·j)
# hoặc DP từ điển (g1, g2), đáp án Σ dp[(g, g)], g ≥ 1
```

**Checklist:**

- [ ] Cặp **có thứ tự**?
- [ ] Loại dãy rỗng (`g = 0`)?
- [ ] Rời nhau (3 nhãn, không đếm độc lập)?
- [ ] `lcm > M` ⇒ `both = 0`?
- [ ] Modulo âm được đưa về dương ở cuối?